# Get hourly rooftop PV generation profile

This script is to extract the PV generation profile per building in Swiwtzerland.

1. find the EGID of the building (Swiss topo)(https://map.geo.admin.ch/#/map?lang=en&center=2660799.87,1190000&z=1&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.pixelkarte-farbe)
2. find the roof information based on EGID (search the data)
3. find the hourly PV profile (in 2016) based on the gemeinde ID, roof aspect, and roof tilt.
4. get the hourly profile

Data source:


Reference:
- Journal article: [Big data mining for the estimation of hourly rooftop photovoltaic potential and its uncertainty](https://doi.org/10.1016/j.apenergy.2019.114404)
- Zenodo:
  - [Global tilted radiation on Swiss rooftops - 2016)](https://zenodo.org/records/4770483)
  - [Rooftop photovoltaic (PV) potential data for the Swiss building stock](https://zenodo.org/records/3609833)

In [1]:
import pandas as pd

## step 1: find the EGID of the building

In [73]:
import sys
import os

# Add the parent directory or specific folder to the path
sys.path.append(os.path.join(os.getcwd(), '..'))  # for parent directory

from swiss_building_tools import get_bldg_attrs_from_address, get_egid_house_no_from_request, get_df_by_key

In [52]:
li_bldgs= []

In [53]:
di_address = {
    'street': 'Gernstrasse',
    'house_no': '1',
    'postal_code': '8311',
    'municipality': 'Brütten'
}


## get the building attributes from address
address = f"{di_address['street']} {di_address['house_no']} {di_address['postal_code']} {di_address['municipality']}"
response = get_bldg_attrs_from_address(address)
li_egid, li_house_no, li_skip = get_egid_house_no_from_request(response, di_address['house_no'], address)

## output
print(f'Number of matched EGID: {len(li_egid)}')
di_address['EGID'] = li_egid[0]
print(f'EGID of the first response: {di_address["EGID"]}')

li_bldgs.append(di_address)

Number of matched EGID: 1
EGID of the first response: 201021879


In [54]:
di_address = {
    'street': 'Zwinggartenstrasse',
    'house_no': '2',
    'postal_code': '8600',
    'municipality': 'Dübendorf'
}

## get the building attributes from address
address = f"{di_address['street']} {di_address['house_no']} {di_address['postal_code']} {di_address['municipality']}"
response = get_bldg_attrs_from_address(address)
li_egid, li_house_no, li_skip = get_egid_house_no_from_request(response, di_address['house_no'], address)

## output
print(f'Number of matched EGID: {len(li_egid)}')
di_address['EGID'] = li_egid[0]
print(f'EGID of the first response: {di_address["EGID"]}')

li_bldgs.append(di_address)

Number of matched EGID: 1
EGID of the first response: 11514299


In [55]:
## create the dataframe of buildings to be processed
# can be multiple buildings

df_bldg = pd.DataFrame(li_bldgs)
df_bldg

,street,house_no,postal_code,municipality,EGID
0,Gernstrasse,1,8311,Brütten,201021879
1,Zwinggartenstrasse,2,8600,Dübendorf,11514299


In [56]:
## find the gemeinde id (GDENR)

# load the gemeinde mapping file
df_gemeinde_map = pd.read_csv('data/gemeinde_map.csv')

# find the gemeinde number
for gemeinde in df_bldg['municipality'].unique():

    df_gde = df_gemeinde_map[df_gemeinde_map['GDENAME']==gemeinde]

    if len(df_gde) == 0:
        print(f'Error: no gemeinde found for {gemeinde}. Please check the name.')
        continue
    elif len(df_gde) == 1:
        gdenr = df_gde.iloc[0]['GDENR']
    elif len(df_gde) > 1:
        print(f'Warning: multiple gemeinde found for {gemeinde}. Using the first one.')
        gdenr = df_gde.iloc[0]['GDENR']

    df_bldg.loc[df_bldg['municipality'] == gemeinde, 'GDENR'] = str(int(gdenr))

df_bldg

,street,house_no,postal_code,municipality,EGID,GDENR
0,Gernstrasse,1,8311,Brütten,201021879,213
1,Zwinggartenstrasse,2,8600,Dübendorf,11514299,191


## step 2: find the roofid

In [49]:
## load EGID rooftop ID mapping
df_uid = pd.read_csv('data/rooftop_link_info_v1_short.csv')
df_uid['EGID'] = df_uid['EGID'].astype(str)

df_uid

,DF_UID,EGID,ROOF_AREA,ROOF_TILT,ROOF_ASPECT
0,1,245033426,15.9,37,10
1,2,245033426,13.7,44,-170
2,3,245033426,63.4,12,-80
3,4,245033426,38.4,33,-80
4,5,245033426,36.4,35,100
...,...,...,...,...,...
9639226,9890019,302046285,307.1,33,168
9639227,9890020,3169556,25.5,4,77
9639228,9890021,302046622,20.2,7,-174
9639229,9890022,302046622,264.7,7,-174


In [57]:
# get the roof info of the building

df_uid_filtered = df_uid.merge(df_bldg, left_on='EGID', right_on='EGID', how='inner')

df_uid_filtered

,DF_UID,EGID,ROOF_AREA,ROOF_TILT,ROOF_ASPECT,street,house_no,postal_code,municipality,GDENR
0,690633,11514299,291.0,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
1,690634,11514299,67.7,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
2,690635,11514299,71.5,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
3,690636,11514299,14.6,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
4,690637,11514299,678.0,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
5,690638,11514299,110.5,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
6,690639,11514299,19.3,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191
7,3366914,201021879,283.4,39,154,Gernstrasse,1,8311,Brütten,213
8,3366915,201021879,323.5,34,-26,Gernstrasse,1,8311,Brütten,213


## step 3: convert ROOF_TILT and ROOF_ASPECT into the required class

roof_tilts = [0, 10, 20, 30, 40, 50]
roof_aspects = ['N', 'E', 'SE', 'S', 'SW', 'W', 'flat']

Aspect angle (in degrees), rounded to multiples of 10° (in CW deg. from south, 
flat areas labelled as “flat”)

In [58]:
from swiss_pv import  get_roof_aspect_tilt_classes

In [59]:
df_roofs = get_roof_aspect_tilt_classes(df_uid_filtered)
df_roofs

,DF_UID,EGID,ROOF_AREA,ROOF_TILT,ROOF_ASPECT,street,house_no,postal_code,municipality,GDENR,ROOF_TILT_CLASS,ROOF_ASPECT_CLASS
0,690633,11514299,291.0,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
1,690634,11514299,67.7,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
2,690635,11514299,71.5,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
3,690636,11514299,14.6,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
4,690637,11514299,678.0,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
5,690638,11514299,110.5,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
6,690639,11514299,19.3,0,0,Zwinggartenstrasse,2,8600,Dübendorf,191,0,flat
7,3366914,201021879,283.4,39,154,Gernstrasse,1,8311,Brütten,213,40,N
8,3366915,201021879,323.5,34,-26,Gernstrasse,1,8311,Brütten,213,30,SE


In [88]:
df_roofs_group = get_df_by_key(df_roofs, 'roof', by=['EGID', 'ROOF_ASPECT_CLASS', 'ROOF_TILT_CLASS'], separate='; ',
                               col_sum=['ROOF_AREA'], col_join=['ROOF_AREA', 'DF_UID'])
df_roofs_group.reset_index(inplace=True)
df_roofs_group

,EGID,ROOF_ASPECT_CLASS,ROOF_TILT_CLASS,count.roof,sum.ROOF_AREA,join.ROOF_AREA,join.DF_UID
0,11514299,flat,0,7,1252.6,291.0; 67.7; 71.5; 14.6; 678.0; 110.5; 19.3,690633; 690634; 690635; 690636; 690637; 690638...
1,201021879,N,40,1,283.4,283.4,3366914
2,201021879,SE,30,1,323.5,323.5,3366915


## step 4: get the hourly solar irradiance profile

In [60]:
folder_path = r'K:\EnergySystemsDatabase\03-1 EPFL\PV Alina Walch\hourly potential\2016_commune_hourly_aggregates_EW\reduced_by_municipality\\'

# load hourly profile per roof id
df_prof_all = pd.DataFrame(index=pd.date_range(start='2016-01-01 00:00:00', end='2016-12-31 23:00:00', freq='h'))
di_egid_roofs = {}

for indx in df_roofs.index:
    row = df_roofs.loc[indx]
    egid = row['EGID']
    gdenr = row['GDENR']

    col_roof = f'{row["ROOF_ASPECT_CLASS"]}_{row["ROOF_TILT_CLASS"]}'

    col_prof = f'{gdenr}_{col_roof}'

    # save roof profile in dic
    if egid not in di_egid_roofs:
        di_egid_roofs[egid] = []
    if col_prof not in di_egid_roofs[egid]:
        di_egid_roofs[egid].append(col_prof)


    if col_prof in df_prof_all.columns:
        print(f'Profile for {col_prof} already loaded.')
        continue

    # load the profile file
    file_path = f'{gdenr}\\commune_hourly_gt.csv'
    df_prof_gf = pd.read_csv(folder_path + file_path, index_col=0, parse_dates=True)

    # save to the df_prof_all
    df_prof_all = pd.concat([df_prof_all, df_prof_gf[[col_roof]]], axis=1)
    df_prof_all.rename(columns={col_roof:col_prof}, inplace=True)

df_prof_all.fillna(0, inplace=True)

df_prof_all

Profile for 191_flat_0 already loaded.
Profile for 191_flat_0 already loaded.
Profile for 191_flat_0 already loaded.
Profile for 191_flat_0 already loaded.
Profile for 191_flat_0 already loaded.
Profile for 191_flat_0 already loaded.


,191_flat_0,213_N_40,213_SE_30
2016-01-01 00:00:00,0.0,0.0,0.0
2016-01-01 01:00:00,0.0,0.0,0.0
2016-01-01 02:00:00,0.0,0.0,0.0
2016-01-01 03:00:00,0.0,0.0,0.0
2016-01-01 04:00:00,0.0,0.0,0.0
...,...,...,...
2016-12-31 19:00:00,0.0,0.0,0.0
2016-12-31 20:00:00,0.0,0.0,0.0
2016-12-31 21:00:00,0.0,0.0,0.0
2016-12-31 22:00:00,0.0,0.0,0.0


## step 5: export the solar irradiance profiles per EGID

In [89]:
for egid in di_egid_roofs:
    df_prof_egid = df_prof_all[di_egid_roofs[egid]]
    df_prof_egid.to_csv(f'export/solar_irr_{egid}.csv')

    df_roofs_egid = df_roofs_group[df_roofs_group['EGID'] == egid]
    df_roofs_egid.to_csv(f'export/roof_info_{egid}.csv')

In [32]:
import plotly.express as px

fig = px.line(df_prof_all, x=df_prof_all.index, y=df_prof_all.columns, 
              title='Hourly solar irradiance profile 2016',
              labels={'value': 'Solar Irradiance (W/m²)', 'variable': 'Roof Configuration'})
fig.show()